# 03 · Transformer models

Fine-tunes five pretrained transformers with **identical settings** (same split, epochs, learning rate, class weighting, max length) so the comparison is fair. Runs locally in VS Code.

| Model | Size | Why it's in the list |
|---|---|---|
| `distilbert-base-uncased` | 66M | small and fast, a lower bound for transformers |
| `bert-base-uncased` | 110M | the classic baseline |
| `roberta-base` | 125M | usually stronger than BERT on reviews |
| `microsoft/deberta-v3-base` | 184M | often tops text-classification benchmarks |
| `cardiffnlp/twitter-roberta-base-sentiment-latest` | 125M | already pretrained for 3-class sentiment |

## Pick a preset

The notebook detects your hardware (NVIDIA CUDA, Apple Silicon MPS, or CPU) and picks a preset. You can override `PRESET` in the settings cell.

| Preset | Training data | Max length | Epochs | Relative cost |
|---|---|---|---|---|
| `full` (default with an NVIDIA GPU) | all ~33,900 reviews | 256 | 3 | 1x on a GPU; about a day+ on a CPU |
| `lite` (default on CPU / Apple Silicon) | 3,000 reviews (1,000 per class) | 64 | 1 | about 1/15 of `cpu` |
| `medium` | 9,000 reviews (3,000 per class) | 64 | 1 | about 3x `lite`; for retraining the best lite models |
| `cpu` | 9,000 reviews (3,000 per class) | 128 | 2 | heavy for a laptop: many hours |
| `smoke` | 600 reviews | 64 | 1 | minutes, only checks that everything runs |

On a Windows laptop CPU, `lite` took about 20 min for DistilBERT and about 40 min each for the RoBERTa and BERT models; DeBERTa is the slowest (expect roughly 1 to 1.5 h). Laptops vary a lot, so watch the first progress bar (`it/s`) for your real speed. Most reviews are short (median 20 words), so a 64-token limit keeps most of each review.

## What goes into the model

* **Input:** the review title (when the reviewer wrote one) + the review text, the same input the classical models in notebook 02 use. Amazon's automatic titles like "Five Stars" were blanked in notebook 01 because they give the label away.
* **Validation set with the real class mix:** 1,500 training reviews are held out *before* the per-class sampling, so they keep the natural 81/8/11 mix. They are used for early stopping and for the cutoff adjustment below. The test set is never used for either.
* **Cutoff adjustment:** balanced training makes the model over-predict neutral on real data. After training, a small offset per class is chosen on the validation set to maximise macro-F1, then applied unchanged to the test set. The result file stores both scores (`macro_f1` with the adjustment, `macro_f1_untuned` without).

Every preset scores on the **full** test set, so the numbers stay comparable with the classical models. Results record which preset produced them, and notebook 04 flags anything that isn't `full`.

Each model's result is saved to `Output/results/` as soon as it finishes, and models already done with the same preset are skipped on re-run. The model in progress saves a checkpoint every half epoch, so if VS Code closes or the laptop sleeps, re-running continues from the last checkpoint instead of starting that model over.

In [1]:
# Installs into the SAME Python this notebook's kernel uses (a terminal `pip` can point at a different Python on Windows).
# After it finishes the first time, restart the kernel (Restart button at the top), then Run All again.
# NVIDIA GPU users: install the CUDA build of PyTorch first, see Readme.
%pip install -q "transformers>=5.0" "datasets>=3.0" "accelerate>=1.0" "sentencepiece>=0.2.1" "protobuf>=4.25" "tiktoken>=0.7" torch

Note: you may need to restart the kernel to use updated packages.


In [2]:
from pathlib import Path
# Works whether the notebook runs from the Notebooks folder (VS Code default) or the project root
ROOT = Path.cwd().parent if Path.cwd().name == 'Notebooks' else Path.cwd()
DATA, OUTPUT = ROOT / 'Data', ROOT / 'Output'
RESULTS = OUTPUT / 'results'
RESULTS.mkdir(parents=True, exist_ok=True)

In [3]:
%pip install datasets torch

In [4]:
import json, time, shutil, gc, math
import numpy as np, pandas as pd, torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForSequenceClassification,
                          TrainingArguments, Trainer, DataCollatorWithPadding, EarlyStoppingCallback)
from sklearn.metrics import classification_report, f1_score
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight

LABELS = ['not satisfied', 'neutral', 'satisfied']
l2i = {l: i for i, l in enumerate(LABELS)}
SEED = 42

# ordered fastest to slowest, so you get results early; put # in front of a line to skip that model
MODELS = {
    'distilbert':      'distilbert-base-uncased',
    'twitter_roberta_sentiment': 'cardiffnlp/twitter-roberta-base-sentiment-latest',
    'roberta_base':    'roberta-base',
    'bert_base':       'bert-base-uncased',
    'deberta_v3_base': 'microsoft/deberta-v3-base',
}

# DeBERTa-v3 ships only a SentencePiece tokenizer; without these packages transformers fails with a confusing 'tiktoken' error
import importlib
missing = []
for module, package in [('sentencepiece', 'sentencepiece'), ('google.protobuf', 'protobuf')]:
    try: importlib.import_module(module)
    except ImportError: missing.append(package)
import sys
print('kernel python:', sys.executable)
if missing:
    raise ImportError(f"{missing} not installed in this kernel's Python. Run the %pip cell above, restart the kernel, then Run All.")

device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'

PRESET = 'auto'   # 'auto', 'full', 'lite', 'medium', 'cpu' or 'smoke'
PRESETS = {
    'full':  dict(per_class=None, max_len=256, epochs=3, batch=32),
    'cpu':   dict(per_class=3000, max_len=128, epochs=2, batch=16),
    'lite':  dict(per_class=1000, max_len=64,  epochs=1, batch=16),
    'medium': dict(per_class=3000, max_len=64, epochs=1, batch=16),   # ~3x lite: for retraining the lite winners
    'smoke': dict(per_class=200,  max_len=64,  epochs=1, batch=16),
}
if PRESET == 'auto':
    PRESET = 'full' if device == 'cuda' else 'lite'
SETTINGS = {**PRESETS[PRESET], 'lr': 2e-5}
print(f'device: {device} | preset: {PRESET} | {SETTINGS}')

def save_result(name, family, y_true, y_pred, extra=None):
    """One JSON per model in Output/results, read by notebook 04."""
    r = classification_report(y_true, y_pred, labels=LABELS, output_dict=True, zero_division=0)
    out = {'model': name, 'family': family, 'macro_f1': r['macro avg']['f1-score'], 'accuracy': r['accuracy'],
           **{f'recall_{l}': r[l]['recall'] for l in LABELS}, **{f'f1_{l}': r[l]['f1-score'] for l in LABELS}, **(extra or {})}
    json.dump(out, open(RESULTS / f'{name}.json', 'w'), indent=2)
    return out

def model_input(df):
    """Review title (when the reviewer wrote one) + text. Notebook 01 already blanked Amazon's automatic 'Five Stars' titles."""
    if 'title' not in df:
        raise ValueError('Data/train.csv has no title column: re-run notebook 01 first (same split, just adds titles).')
    title, text = df['title'].fillna('').astype(str).str.strip(), df['text'].astype(str)
    return np.where(title != '', title + ' . ' + text, text).tolist()

def tune_offsets(scores, y):
    """Per-class cutoff adjustment: add an offset to each class score, chosen to maximise macro-F1 on held-out data.
    scores: n x 3 array in LABELS order (log-probabilities or decision scores); y: true label indices."""
    grid = np.round(np.arange(-1.5, 1.51, 0.1), 2)
    best_f1, best = f1_score(y, scores.argmax(1), average='macro'), np.zeros(3)
    for a in grid:
        for b in grid:
            off = np.array([a, b, 0.0])   # not satisfied, neutral; satisfied is the reference
            f = f1_score(y, (scores + off).argmax(1), average='macro')
            if f > best_f1 + 1e-9:
                best_f1, best = f, off
    return best

device: cpu | preset: lite | {'per_class': 1000, 'max_len': 64, 'epochs': 1, 'batch': 16, 'lr': 2e-05}


In [5]:
train_full = pd.read_csv(DATA / 'train.csv').dropna(subset=['text'])
test = pd.read_csv(DATA / 'test.csv').dropna(subset=['text'])   # always the full test set
if PRESET == 'smoke':
    test = test.sample(500, random_state=SEED)
# Validation set held out FIRST, so it keeps the real 81/8/11 class mix: used for early stopping and cutoff tuning.
# The test set is only touched once per model, at the end.
rest, val = train_test_split(train_full, test_size=300 if PRESET == 'smoke' else 1500,
                             stratify=train_full.label, random_state=SEED)
train = rest
if SETTINGS['per_class']:   # balanced sample for training only
    train = pd.concat([g.sample(min(len(g), SETTINGS['per_class']), random_state=SEED) for _, g in rest.groupby('label')])
print(f'train {len(train)} {train.label.value_counts().to_dict()} | val {len(val)} (natural mix) | test {len(test)}')

# class-weighted loss to counter the 80/12/8 imbalance (sqrt-dampened: full weights over-predict neutral)
cw = torch.tensor(np.sqrt(compute_class_weight('balanced', classes=np.arange(3), y=train.label.map(l2i).values)), dtype=torch.float)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop('labels')
        out = model(**inputs)
        loss = torch.nn.functional.cross_entropy(out.logits, labels, weight=cw.to(out.logits.device))
        return (loss, out) if return_outputs else loss

def metrics(p):
    return {'macro_f1': f1_score(p.label_ids, p.predictions.argmax(-1), average='macro')}

train 3000 {'neutral': 1000, 'not satisfied': 1000, 'satisfied': 1000} | val 1500 (natural mix) | test 8840


In [6]:
INPUT = 'title+text'

def result_name(name):
    # the preset is part of the name, so lite / medium / full results sit side by side and never overwrite each other
    return f'{name}_{PRESET}'

def already_done(name):
    f = RESULTS / f'{result_name(name)}.json'
    if not f.exists(): return False
    r = json.load(open(f))
    return r.get('preset') == PRESET and r.get('input') == INPUT

def run(name, checkpoint):
    if already_done(name):
        print(f'{name}: already done with preset {PRESET}, skipping'); return
    t = time.time()
    tok = AutoTokenizer.from_pretrained(checkpoint)
    def to_ds(df):
        ds = Dataset.from_dict({'text': model_input(df), 'label': df.label.map(l2i).tolist()})
        return ds.map(lambda b: tok(b['text'], truncation=True, max_length=SETTINGS['max_len']), batched=True, remove_columns=['text'])
    ds_train, ds_val, ds_test = to_ds(train), to_ds(val), to_ds(test)

    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint, num_labels=3, id2label=dict(enumerate(LABELS)), label2id=l2i, ignore_mismatched_sizes=True,
        dtype=torch.float32)   # some checkpoints (DeBERTa-v3) are stored in fp16; train in fp32 or CPU errors with Half vs Float
    ckpt_dir = OUTPUT / 'checkpoints' / f'{name}_{PRESET}'   # per preset, so a checkpoint is never resumed with different settings
    # checkpoint every half epoch so a crash or closed window loses at most half an epoch
    every = max(20, math.ceil(len(ds_train) / SETTINGS['batch']) // 2)
    args = TrainingArguments(
        output_dir=str(ckpt_dir), num_train_epochs=SETTINGS['epochs'], learning_rate=SETTINGS['lr'],
        weight_decay=0.01, warmup_steps=0.06,
        per_device_train_batch_size=SETTINGS['batch'], per_device_eval_batch_size=64,
        eval_strategy='steps', eval_steps=every, save_strategy='steps', save_steps=every,
        load_best_model_at_end=True, metric_for_best_model='macro_f1', save_total_limit=2,
        fp16=(device == 'cuda' and 'deberta' not in checkpoint),   # DeBERTa-v3 can go NaN in fp16
        dataloader_num_workers=0, report_to='none', seed=SEED)
    trainer = WeightedTrainer(model=model, args=args, train_dataset=ds_train, eval_dataset=ds_val,
                              processing_class=tok, data_collator=DataCollatorWithPadding(tok),
                              compute_metrics=metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    resume = any(ckpt_dir.glob('checkpoint-*'))
    if resume:
        print(f'{name}: resuming from the last saved checkpoint')
    trainer.train(resume_from_checkpoint=True if resume else None)

    # cutoff adjustment: offsets tuned on the natural-mix validation set, then applied unchanged to test
    log_probs = lambda ds: torch.log_softmax(torch.tensor(trainer.predict(ds).predictions, dtype=torch.float32), -1).numpy()
    offsets = tune_offsets(log_probs(ds_val), val.label.map(l2i).values)
    test_scores = log_probs(ds_test)
    y_true = test.label.map(l2i).values
    y_pred = (test_scores + offsets).argmax(-1)
    f1_untuned = f1_score(y_true, test_scores.argmax(-1), average='macro')
    trainer.save_model(OUTPUT / 'models' / result_name(name)); tok.save_pretrained(OUTPUT / 'models' / result_name(name))
    r = save_result(result_name(name), 'transformer', [LABELS[i] for i in y_true], [LABELS[i] for i in y_pred],
                    {'checkpoint': checkpoint, 'preset': PRESET, 'input': INPUT, 'macro_f1_untuned': f1_untuned,
                     'offsets': offsets.tolist(), 'device': device, 'train_seconds': round(time.time() - t)})
    print(f'{result_name(name)}: macro-F1 {f1_untuned:.3f} -> {r["macro_f1"]:.3f} with cutoff adjustment  ({(time.time() - t) / 60:.0f} min)')

    shutil.rmtree(ckpt_dir, ignore_errors=True)
    del model, trainer; gc.collect()
    if device == 'cuda': torch.cuda.empty_cache()

RUN_ONLY = []   # e.g. ['twitter_roberta_sentiment', 'roberta_base'] to train just those; empty = all MODELS
failed = {}
for name, checkpoint in MODELS.items():
    if RUN_ONLY and name not in RUN_ONLY:
        continue
    try:
        run(name, checkpoint)
    except Exception as e:   # one broken model shouldn't stop the rest; re-run later to retry it
        failed[name] = f'{type(e).__name__}: {e}'
        print(f'{name}: FAILED, skipping for now -> {failed[name][:300]}')
if failed:
    print(); print('Re-run this cell after fixing the error; finished models are skipped:', list(failed))

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1500 [00:00<?, ? examples/s]

Map:   0%|          | 0/8840 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


...

Step,Training Loss,Validation Loss,Macro F1
94,No log,0.375720,0.694312
188,No log,0.382919,0.698140


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

deberta_v3_base_lite: macro-F1 0.704 -> 0.719 with cutoff adjustment  (73 min)
